In [1]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.chat_models import init_chat_model
from langchain_ollama import OllamaEmbeddings
# langchain中华的pdf解析器
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()
# ===============一、构建知识库阶段====================
# =============1.加载文档===============
# 通过解析器类创建对象
loader = PyPDFLoader(file_path="resources/贵州茅台研报.pdf", mode="single")
# 通过load得到文档
docs = loader.load()

# =============2.切分文档===============
# 导入文档切分类,得到切分对象
splitter = CharacterTextSplitter(separator="\n", chunk_size=800, chunk_overlap=150)
# 调用对象身上的切分文档方法,传入文档对象
chunks = splitter.split_documents(docs)
print(f"分块数量：{len(chunks)}")


# =============3.向量化===============
# 向量模型，这里用阿里做的本地向量化模型,使用Ollama加载
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=1024)


# =============4.存入向量库===============
# chunks里面是切分好的文档片段,一个列表,导入向量模型
# 使用基于内存存储的向量化数据库
vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)

# ==============二、在线问答阶段================
# 创建模型
model = init_chat_model(
    "deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


def try_rag(query: str):
    # 1.检索文档（VectorStore会自动把问题向量化，召回相关知识片段）
    # 将问题传入向量化数据库对象,进行语义相似度匹配
    retrieved_docs = vectorstore.similarity_search(query, k=2)

    # 2.拼接上下文提示词
    # 从向量数据库中检索出的文档片段,拼接进提示词
    content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # 编写提示词
    prompt = f"""你基于我提供的报告回答用户问题，报告中没提及的就说不知道，不要自己编造答案.
    report: ```{content}```
    query: {query}"""
    # 3.调用模型，生成答案
    response = model.invoke(prompt)
    return response.content


# ================三、测试=================
print(try_rag("茅台收盘价多少"))
print('=' * 100)
print(try_rag("茅台2025年的市盈率和市净率是多少"))

C:\Users\84370\Desktop\黑马-python阶段\python-project\wcy-insurance\agent-service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\84370\AppData\Local\Temp\ipykernel_26088\4050643397.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


分块数量：14
根据您提供的报告，贵州茅台的收盘价为1,458.49元（数据日期为2026年4月24日）。
根据您提供的报告，其中仅包含2025A/E和2026E的市盈率（PE）数据，未提及市净率（PB）相关信息，因此无法回答茅台的市净率。2025年的市盈率（PE）为22.2倍。
